
# **Epidemic Modeling**

This notebook demonstartes the followings:

1. SIR model
2. The basic reproduction number $R_0$ and outbreak conditions
3. Vaccination and herd immunity
4. SEIR model
5. Stochastic SIR
6. SIR on a contact network



## Imports

Run this cell to import the required libraries and set some plotting defaults.


In [ ]:
# imports
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import networkx as nx

# show plots inline
%matplotlib inline

plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True

N = 1000  # default population size


## **SIR model**

We consider the classic SIR model:

\begin{align}
\frac{dS}{dt} &= -\beta \frac{S I}{N} \\
\frac{dI}{dt} &= \beta \frac{S I}{N} - \gamma I \\
\frac{dR}{dt} &= \gamma I
\end{align}

where
- $S(t)$ = number of susceptible individuals  
- $I(t)$ = number of infectious individuals  
- $R(t)$ = number of recovered (and immune) individuals  
- $N = S + I + R$ is the total population  
- $\beta$ is the transmission rate  
- $\gamma$ is the recovery rate  


### **SIR**: Exercise 1

1. Choose values for $\beta$ and $\gamma$.  
2. Implement the SIR model of ODEs using the function `sir(t, y)` that returns `[dSdt, dIdt, dRdt]`.  
3. Use `scipy.integrate.solve_ivp` to simulate the system over 100 days.  
4. Plot $S(t)$, $I(t)$, and $R(t)$ on the same axes.

Questions:
- When does the infection peak?  
- Does everyone eventually get infected?


In [ ]:
# Exercise 1

# Parameters
beta = 0.3      # transmission rate
gamma = 0.1     # recovery rate

# Initial conditions
I0 = 1
R0 = 0
S0 = N - I0 - R0
y0 = (S0, I0, R0)

def sir(t, y):
    S, I, R = y
    dSdt = -beta * S * I / N
    dIdt = beta * S * I / N - gamma * I
    dRdt = gamma * I
    return [dSdt, dIdt, dRdt]

t_span = (0, 100)
t_eval = np.linspace(t_span[0], t_span[1], 500)

solution = solve_ivp(sir, t_span, y0, t_eval=t_eval)
S, I, R = solution.y

plt.figure()
plt.plot(t_eval, S, label="Susceptible")
plt.plot(t_eval, I, label="Infectious")
plt.plot(t_eval, R, label="Recovered")
plt.xlabel("Time")
plt.ylabel("Number of people")
plt.legend()
plt.title("SIR model dynamics")
plt.show()

print("Peak number infected:", I.max())
print("Time of peak infection:", t_eval[I.argmax()])
print("Final number recovered:", R[-1])



---

## **The basic reproduction number $R_0$ and outbreaks**

The *basic reproduction number* $R_0$ for the SIR model in a fully susceptible population is

\begin{equation}
R_0 = \frac{\beta}{\gamma}.
\end{equation}

- If $R_0 > 1$, we expect an outbreak (the number of infectious individuals initially grows).  
- If $R_0 < 1$, infections fade out quickly.


### **SIR**: Exercise 2

1. Fix $\gamma = 0.1$.  
2. Consider several values of $\beta$, e.g. `[0.05, 0.1, 0.2, 0.5]`. For each:
   - Simulate the SIR model.  
   - Plot $I(t)$ on the same axes.  
   - Compute and annotate $R_0 = \beta/\gamma$.
3. Compare the curves and identify which values of $\beta$ lead to an outbreak.

Questions:
- For which values of $R_0$ does $I(t)$ grow initially?  
- How does increasing $R_0$ affect peak size and timing?


In [ ]:

# Exercise 2

gamma = 0.1
beta_values = [0.05, 0.1, 0.2, 0.5]

plt.figure()
for beta_val in beta_values:
    def sir_param(t, y):
        S, I, R = y
        dSdt = -beta_val * S * I / N
        dIdt = beta_val * S * I / N - gamma * I
        dRdt = gamma * I
        return [dSdt, dIdt, dRdt]

    y0 = (N - 1, 1, 0)
    sol = solve_ivp(sir_param, t_span, y0, t_eval=t_eval)
    S_sim, I_sim, R_sim = sol.y

    R0_basic = beta_val / gamma
    plt.plot(t_eval, I_sim, label=f"beta={beta_val:.2f}, R0={R0_basic:.1f}")

plt.xlabel("Time")
plt.ylabel("Number infected")
plt.title("Effect of R0 on infection dynamics")
plt.legend()
plt.show()



---

## **Vaccination and the Herd Immunity Threshold**

Suppose we vaccinate a fraction $p$ of the population *before* the epidemic starts. These people move directly to the *Recovered* compartment (perfect vaccine). The initial susceptible population is then

\begin{equation}
S_0 = (1 - p) N.
\end{equation}

The effective reproduction number at the start is

\begin{equation}
R_{\,\text{eff}} = R_0 (1 - p).
\end{equation}

To prevent an outbreak, we need

\begin{equation}
R_{\,\text{eff}} < 1 \quad \Leftrightarrow \quad p > 1 - \frac{1}{R_0},
\end{equation}

which is the *critical vaccination threshold*.


### **SIR**: Exercise 3

1. Use $\beta = 0.3$, $\gamma = 0.1$ and compute $R_0$ and the critical vaccination coverage $p_c = 1 - 1/R_0$.  
2. For each $p$ in `[0.0, 0.2, 0.4, 0.6, 0.8]`:
   - Set initial conditions with $S_0 = (1 - p)N$, $I_0 = 1$, and $R_0 = pN$.  
   - Simulate the SIR model.  
   - Plot $I(t)$ for all $p$ on the same axis.
3. Compare which vaccination levels prevent a large outbreak.

Questions:
- Does the numerical threshold for preventing the outbreak match the theoretical value $p_c$?

In [ ]:

# Exercise 3

beta = 0.3
gamma = 0.1
R0_basic = beta / gamma
p_c = 1 - 1 / R0_basic

print("R0 =", R0_basic)
print("Critical vaccination coverage p_c =", p_c)

p_values = [0.0, 0.2, 0.4, 0.6, 0.8]

plt.figure()
for p in p_values:
    S0 = (1 - p) * N
    I0 = 1
    R0_init = N - S0 - I0  # equals p*N if N is large and I0 small
    y0_v = (S0, I0, R0_init)

    def sir_vacc(t, y):
        S, I, R = y
        dSdt = -beta * S * I / N
        dIdt = beta * S * I / N - gamma * I
        dRdt = gamma * I
        return [dSdt, dIdt, dRdt]

    sol_v = solve_ivp(sir_vacc, t_span, y0_v, t_eval=t_eval)
    S_v, I_v, R_v = sol_v.y
    plt.plot(t_eval, I_v, label=f"p={p:.1f}")

plt.xlabel("Time")
plt.ylabel("Number infected")
plt.title("Vaccination and outbreak prevention")
plt.legend()
plt.show()


## **SEIR model**

Many infections have a *latent (incubation)* period: people are infected but not yet infectious. We can add an *Exposed* compartment $E$ to the SIR model:

\begin{align}
\frac{dS}{dt} &= -\beta \frac{S I}{N} \\
\frac{dE}{dt} &= \beta \frac{S I}{N} - \sigma E \\
\frac{dI}{dt} &= \sigma E - \gamma I \\
\frac{dR}{dt} &= \gamma I,
\end{align}

where $1/\sigma$ is the *average incubation period*.


### **SEIR**: Exercise 4

1. Implement the SEIR model above as a function `seir(t, y)` that returns `[dSdt, dEdt, dIdt, dRdt]`.  
2. Use parameters, for example:
   - $\beta = 0.3$  
   - $\gamma = 0.1$  
   - $\sigma = 1/5$ (5 days incubation)  
3. Plot $S(t)$, $E(t)$, $I(t)$, and $R(t)$.  
4. Compare the infection curve to the SIR model (e.g. by overlaying them or by visually comparing shapes).

Questions:
- Does an incubation period delay the epidemic peak?  
- How does it affect peak height?


In [ ]:

# Exercise 4

beta = 0.3
gamma = 0.1
sigma = 1 / 5.0   # average 5 time units incubation

# Initial conditions
E0 = 0
I0 = 1
R0_init = 0
S0 = N - E0 - I0 - R0_init
y0_seir = (S0, E0, I0, R0_init)

def seir(t, y):
    S, E, I, R = y
    dSdt = -beta * S * I / N
    dEdt = beta * S * I / N - sigma * E
    dIdt = sigma * E - gamma * I
    dRdt = gamma * I
    return [dSdt, dEdt, dIdt, dRdt]

sol_seir = solve_ivp(seir, t_span, y0_seir, t_eval=t_eval)
S_seir, E_seir, I_seir, R_seir = sol_seir.y

plt.figure()
plt.plot(t_eval, S_seir, label="S (SEIR)")
plt.plot(t_eval, E_seir, label="E (SEIR)")
plt.plot(t_eval, I_seir, label="I (SEIR)")
plt.plot(t_eval, R_seir, label="R (SEIR)")
plt.xlabel("Time")
plt.ylabel("Number of people")
plt.title("SEIR model dynamics")
plt.legend()
plt.show()

# For comparison: SIR with same beta, gamma
def sir_compare(t, y):
    S, I, R = y
    dSdt = -beta * S * I / N
    dIdt = beta * S * I / N - gamma * I
    dRdt = gamma * I
    return [dSdt, dIdt, dRdt]

y0_sir = (N - 1, 1, 0)
sol_sir_comp = solve_ivp(sir_compare, t_span, y0_sir, t_eval=t_eval)
S_cmp, I_cmp, R_cmp = sol_sir_comp.y

plt.figure()
plt.plot(t_eval, I_cmp, label="I (SIR)")
plt.plot(t_eval, I_seir, label="I (SEIR)")
plt.xlabel("Time")
plt.ylabel("Number infected")
plt.title("Comparison: SIR vs SEIR infection curves")
plt.legend()
plt.show()


## **Stochastic SIR**

The ODE SIR model is *deterministic* and assumes a very large population where random fluctuations are negligible. In reality, especially when there are only a few infectious individuals, chance plays a big role. A *stochastic SIR* model represents infection and recovery as random events.

We can build a simple discrete-time, stochastic SIR model:

- Time step: $\Delta t$  
- Infection probability for each susceptible: roughly $\beta \frac{I}{N} \Delta t$  
- Recovery probability for each infectious: $\gamma \Delta t$

At each step:

- New infections $\sim \text{Binomial}(S, p_\text{inf})$  
- New recoveries $\sim \text{Binomial}(I, p_\text{rec})$

### **Stochastic SIR**: Exercise 5

1. Implement a function `stochastic_sir(T, dt)` that returns time series of $(S, I, R)$.  
2. Run the simulation many times (e.g. 20 realizations) and plot $I(t)$ for each run on the same figure.  
3. Compare the stochastic curves with the deterministic solution.

Think:
- Do some runs fade out early even when $R_0 > 1$?  
- How variable are peak sizes and timings?


In [ ]:

# Exercise 5

def stochastic_sir(T=160, dt=0.2, beta=0.3, gamma=0.1, N=1000):
    steps = int(T / dt)
    S = np.zeros(steps)
    I = np.zeros(steps)
    R = np.zeros(steps)

    S[0] = N - 1
    I[0] = 1
    R[0] = 0

    for t in range(1, steps):
        p_inf = beta * I[t - 1] / N * dt
        p_rec = gamma * dt

        # Bound probabilities between 0 and 1
        p_inf = min(max(p_inf, 0.0), 1.0)
        p_rec = min(max(p_rec, 0.0), 1.0)

        new_inf = np.random.binomial(int(S[t - 1]), p_inf)
        new_rec = np.random.binomial(int(I[t - 1]), p_rec)

        S[t] = S[t - 1] - new_inf
        I[t] = I[t - 1] + new_inf - new_rec
        R[t] = R[t - 1] + new_rec

    times = np.linspace(0, T, steps)
    return times, S, I, R

# Run multiple realizations
T = 160
dt = 0.2
n_runs = 20

plt.figure()
for _ in range(n_runs):
    times, S_st, I_st, R_st = stochastic_sir(T=T, dt=dt, beta=0.3, gamma=0.1, N=N)
    plt.plot(times, I_st, alpha=0.3)

plt.xlabel("Time")
plt.ylabel("Number infected")
plt.title("Stochastic SIR")
plt.show()



---

## **SIR on a contact network**

The SIR model assumes *homogeneous mixing*: everyone is equally likely to contact everyone else. A more realistic representation uses a contact network:

- Nodes represent individuals.  
- Edges represent contacts along which infection can spread.

We can simulate SIR on a network by:

1. Generating a graph (e.g. Erdős-Rényi random graph for simplicity).  
2. At each time step:
   - Susceptible nodes with infectious neighbors can become infected.  
   - Infectious nodes can recover.

### **SIR on a contact network:** Exercise 6

1. Build an Erdős–Rényi graph with `N_nodes = 500` and edge probability `p_edge = 0.01`.  
2. Implement a simple SIR process on the network.  
3. Plot the number of infectious nodes over time.

Questions:
- How does the epidemic curve compare to the SIR model?  
- How might changing the average degree (by changing `p_edge`) affect spread?


In [ ]:
# Exercise 6

N_nodes = 500
p_edge = 0.01
beta_net = 0.3
gamma_net = 0.1
T_steps = 80

G = nx.erdos_renyi_graph(N_nodes, p_edge)
# G = nx.karate_club_graph()
# N_nodes = len(G.nodes())

# 0 = S, 1 = I, 2 = R
state = np.zeros(N_nodes, dtype=int)
initial_infected = np.random.choice(N_nodes)
state[initial_infected] = 1

I_counts = []

for t in range(T_steps):
    new_state = state.copy()
    for i in range(N_nodes):
        if state[i] == 0:  # susceptible
            inf_neighbors = sum(state[j] == 1 for j in G.neighbors(i))
            if inf_neighbors > 0:
                # Probability at least one infected neighbor transmits
                p_inf = 1.0 - (1.0 - beta_net) ** inf_neighbors
                if np.random.rand() < p_inf:
                    new_state[i] = 1
        elif state[i] == 1:  # infectious
            if np.random.rand() < gamma_net:
                new_state[i] = 2
    state = new_state
    I_counts.append(np.sum(state == 1))

plt.figure()
plt.plot(range(T_steps), I_counts)
plt.xlabel("Time step")
plt.ylabel("Number infected")
plt.title("SIR on an contact network")
plt.show()
